# 18 XGBoost Optuna mit target encoded ps_car_11_cat

## Import

In [11]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import TargetEncoder

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from optuna_integration import XGBoostPruningCallback

from xgboost import XGBClassifier

import matplotlib as mpl
import matplotlib.pyplot as plt

In [12]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [13]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [14]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)]
    }
)

train    [476168, 476168, 476168]
val         [59522, 59522, 59522]
test        [59522, 59522, 59522]
full     [595212, 595212, 595212]
dtype: object

## Hilfsvariablen

In [15]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [16]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_val": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

In [17]:
x_train_no_calc = x_train[feat_cols_no_calc]
x_val_no_calc = x_val[feat_cols_no_calc]
x_test_no_calc = x_test[feat_cols_no_calc]
x_full_no_calc = x_full[feat_cols_no_calc]

## Optuna Hilfe

## Suchräume

|Parameter|Bereich|
|---|---|
|learning_rate|0.01-0.1 (log)|
|max_depth|3-7|
|min_child_weight|1-200 (log)|
|gamma|1e-3-5 (log)|
|reg_lambda|0.1-100 (log)|
|reg_alpha|1e-3-10 (log)|
|max_delta_step|0-10|
|subsample|0.5-1.0|
|colsample_bynode|0.5-1.0|
|scale_pos_weight|1 / 5 / 27|
|max_cat_to_onehot|2 / 10 / 18|

In [18]:
fixed_params_xgb = {
    "random_state": RANDOM_STATE,
    "verbosity": 0,
    "device": "cpu",
    "n_jobs": -1,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "enable_categorical": True,
    "n_estimators": 3000,
    "early_stopping_rounds": 100
}

In [19]:
def suchraum_params(trial):
    """Suchraum definition"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 200.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 100.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1, 5, 27]),
        "max_cat_to_onehot": trial.suggest_categorical("max_cat_to_onehot", [2, 10, 18]),
    }

    return params

In [20]:
targ_enc = TargetEncoder(cv=5, random_state=RANDOM_STATE)

x_train_te = x_train.copy()
x_val_te = x_val.copy()
x_test_te = x_test.copy()

x_train_te[high_kard_cols] = targ_enc.fit_transform(x_train[high_kard_cols], y_train)
x_val_te[high_kard_cols] = targ_enc.transform(x_val[high_kard_cols])
x_test_te[high_kard_cols] = targ_enc.transform(x_test[high_kard_cols])

x_train_no_calc_te = x_train_te[feat_cols_no_calc]
x_val_no_calc_te = x_val_te[feat_cols_no_calc]
x_test_no_calc_te = x_test_te[feat_cols_no_calc]

x_train_te[high_kard_cols].describe()

c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


,ps_car_11_cat
count,476168.000000
mean,0.036446
std,0.009676
min,0.016329
25%,0.028362
50%,0.034718
75%,0.044178
max,0.078113


## Mit Calc

In [21]:
def objective_xgb_calc_te(trial):
    pruning_callback = XGBoostPruningCallback(trial, "validation_0-auc")

    modell = XGBClassifier(
        **fixed_params_xgb,
        **suchraum_params(trial),
        callbacks=[pruning_callback]
    )

    modell.fit(x_train_te, y_train, eval_set=[(x_val_te, y_val)], verbose=0)

    return roc_auc_score(y_val, modell.predict_proba(x_val_te)[:, 1])

In [22]:
study_xgb_calc_te = optuna.create_study(
    study_name = "XGBoost_calc_te",
    storage = "sqlite:///xgboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

[I 2026-08-18 15:21:53,462] A new study created in RDB with name: XGBoost_calc_te


In [23]:
study_xgb_calc_te.optimize(
    objective_xgb_calc_te,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

[I 2026-08-18 15:22:20,298] Trial 0 finished with value: 0.6327024093194189 and parameters: {'learning_rate': 0.023688639503640783, 'max_depth': 7, 'min_child_weight': 48.343714531846395, 'gamma': 0.16383993835282307, 'reg_lambda': 0.2938027938703535, 'reg_alpha': 0.004207053950287938, 'max_delta_step': 0, 'subsample': 0.9330880728874675, 'colsample_bynode': 0.8005575058716043, 'scale_pos_weight': 27, 'max_cat_to_onehot': 2}. Best is trial 0 with value: 0.6327024093194189.
[I 2026-08-18 15:24:28,919] Trial 1 finished with value: 0.634489474432188 and parameters: {'learning_rate': 0.015254729458052608, 'max_depth': 4, 'min_child_weight': 16.12427845856261, 'gamma': 0.039605150456850806, 'reg_lambda': 0.7476312062252299, 'reg_alpha': 0.2801635158716261, 'max_delta_step': 1, 'subsample': 0.6460723242676091, 'colsample_bynode': 0.6831809216468459, 'scale_pos_weight': 5, 'max_cat_to_onehot': 10}. Best is trial 1 with value: 0.634489474432188.
[I 2026-08-18 15:25:16,041] Trial 2 finished wit

In [24]:
pd.DataFrame({
    "best auc_val": study_xgb_calc_te.best_value,
    "best gini": 2 * study_xgb_calc_te.best_value - 1,
    "trials": len(study_xgb_calc_te.trials),
    "pruned": len([t for t in study_xgb_calc_te.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_xgb_calc_te.trials if t.state.name == "FAIL"]),
    "best params": study_xgb_calc_te.best_params
})

,best auc_val,best gini,trials,pruned,fail,best params
learning_rate,0.637973,0.275947,50,8,0,0.018784
max_depth,0.637973,0.275947,50,8,0,6.000000
min_child_weight,0.637973,0.275947,50,8,0,26.805781
gamma,0.637973,0.275947,50,8,0,0.004322
reg_lambda,0.637973,0.275947,50,8,0,42.387085
reg_alpha,0.637973,0.275947,50,8,0,0.124704
max_delta_step,0.637973,0.275947,50,8,0,7.000000
subsample,0.637973,0.275947,50,8,0,0.610842
colsample_bynode,0.637973,0.275947,50,8,0,0.674854
scale_pos_weight,0.637973,0.275947,50,8,0,1.000000


In [25]:
study_xgb_calc_te.best_params

{'learning_rate': 0.01878422273943782,
 'max_depth': 6,
 'min_child_weight': 26.80578055959848,
 'gamma': 0.0043219290644965075,
 'reg_lambda': 42.387084879302726,
 'reg_alpha': 0.12470405254321192,
 'max_delta_step': 7,
 'subsample': 0.6108424214140475,
 'colsample_bynode': 0.6748543547526068,
 'scale_pos_weight': 1,
 'max_cat_to_onehot': 2}

## Ohne Calc

In [26]:
def objective_xgb_no_calc_te(trial):
    pruning_callback = XGBoostPruningCallback(trial, "validation_0-auc")

    modell = XGBClassifier(
        **fixed_params_xgb,
        **suchraum_params(trial),
        callbacks=[pruning_callback]
    )

    modell.fit(x_train_no_calc_te, y_train, eval_set=[(x_val_no_calc_te, y_val)], verbose=0)

    return roc_auc_score(y_val, modell.predict_proba(x_val_no_calc_te)[:, 1])

In [27]:
study_xgb_no_calc_te = optuna.create_study(
    study_name = "XGBoost_no_calc_te",
    storage = "sqlite:///xgboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

[I 2026-08-18 16:02:25,512] A new study created in RDB with name: XGBoost_no_calc_te


In [28]:
study_xgb_no_calc_te.optimize(
    objective_xgb_no_calc_te,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

[I 2026-08-18 16:02:47,809] Trial 0 finished with value: 0.6306568727667212 and parameters: {'learning_rate': 0.023688639503640783, 'max_depth': 7, 'min_child_weight': 48.343714531846395, 'gamma': 0.16383993835282307, 'reg_lambda': 0.2938027938703535, 'reg_alpha': 0.004207053950287938, 'max_delta_step': 0, 'subsample': 0.9330880728874675, 'colsample_bynode': 0.8005575058716043, 'scale_pos_weight': 27, 'max_cat_to_onehot': 2}. Best is trial 0 with value: 0.6306568727667212.
[I 2026-08-18 16:04:24,237] Trial 1 finished with value: 0.6364710874651968 and parameters: {'learning_rate': 0.015254729458052608, 'max_depth': 4, 'min_child_weight': 16.12427845856261, 'gamma': 0.039605150456850806, 'reg_lambda': 0.7476312062252299, 'reg_alpha': 0.2801635158716261, 'max_delta_step': 1, 'subsample': 0.6460723242676091, 'colsample_bynode': 0.6831809216468459, 'scale_pos_weight': 5, 'max_cat_to_onehot': 10}. Best is trial 1 with value: 0.6364710874651968.
[I 2026-08-18 16:05:17,058] Trial 2 finished w

In [29]:
pd.DataFrame({
    "best auc_val": study_xgb_no_calc_te.best_value,
    "best gini": 2 * study_xgb_no_calc_te.best_value - 1,
    "trials": len(study_xgb_no_calc_te.trials),
    "pruned": len([t for t in study_xgb_no_calc_te.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_xgb_no_calc_te.trials if t.state.name == "FAIL"]),
    "best params": study_xgb_no_calc_te.best_params
})

,best auc_val,best gini,trials,pruned,fail,best params
learning_rate,0.64047,0.280939,50,15,0,0.068409
max_depth,0.64047,0.280939,50,15,0,4.000000
min_child_weight,0.64047,0.280939,50,15,0,123.572582
gamma,0.64047,0.280939,50,15,0,1.226278
reg_lambda,0.64047,0.280939,50,15,0,1.262450
reg_alpha,0.64047,0.280939,50,15,0,0.389757
max_delta_step,0.64047,0.280939,50,15,0,4.000000
subsample,0.64047,0.280939,50,15,0,0.871835
colsample_bynode,0.64047,0.280939,50,15,0,0.697225
scale_pos_weight,0.64047,0.280939,50,15,0,1.000000


In [30]:
study_xgb_no_calc_te.best_params

{'learning_rate': 0.06840892817037018,
 'max_depth': 4,
 'min_child_weight': 123.5725817719912,
 'gamma': 1.2262777677477144,
 'reg_lambda': 1.2624495151772979,
 'reg_alpha': 0.3897567030114538,
 'max_delta_step': 4,
 'subsample': 0.8718352321453859,
 'colsample_bynode': 0.6972248821194833,
 'scale_pos_weight': 1,
 'max_cat_to_onehot': 10}

## Load Study

In [31]:
study_xgb_calc_te_loaded = optuna.load_study(
    study_name = "XGBoost_calc_te",
    storage = "sqlite:///xgboost_opti.db"
)


In [32]:
study_xgb_no_calc_te_loaded = optuna.load_study(
    study_name = "XGBoost_no_calc_te",
    storage = "sqlite:///xgboost_opti.db"
)

## Best Params again

In [33]:
study_xgb_calc_te_loaded.best_params

{'learning_rate': 0.01878422273943782,
 'max_depth': 6,
 'min_child_weight': 26.80578055959848,
 'gamma': 0.0043219290644965075,
 'reg_lambda': 42.387084879302726,
 'reg_alpha': 0.12470405254321192,
 'max_delta_step': 7,
 'subsample': 0.6108424214140475,
 'colsample_bynode': 0.6748543547526068,
 'scale_pos_weight': 1,
 'max_cat_to_onehot': 2}

In [34]:
study_xgb_no_calc_te_loaded.best_params

{'learning_rate': 0.06840892817037018,
 'max_depth': 4,
 'min_child_weight': 123.5725817719912,
 'gamma': 1.2262777677477144,
 'reg_lambda': 1.2624495151772979,
 'reg_alpha': 0.3897567030114538,
 'max_delta_step': 4,
 'subsample': 0.8718352321453859,
 'colsample_bynode': 0.6972248821194833,
 'scale_pos_weight': 1,
 'max_cat_to_onehot': 10}

## 100% Train

In [35]:
results = []
train_times = {}

In [36]:
name = "L_xgb_opt_03"

L_xgb_opt_03 = XGBClassifier(
    **fixed_params_xgb,
    **study_xgb_calc_te_loaded.best_params
)

start = time.time()
L_xgb_opt_03.fit(
    x_train_te, y_train, 
    eval_set=[(x_val_te, y_val)], 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_xgb_opt_03, x_train_te, y_train, x_val_te, y_val, train_times[name], best_iter=L_xgb_opt_03.best_iteration)
)

[0]	validation_0-auc:0.59660
[250]	validation_0-auc:0.63438
[500]	validation_0-auc:0.63640
[750]	validation_0-auc:0.63777
[882]	validation_0-auc:0.63742


In [37]:
name = "L_xgb_opt_04"

L_xgb_opt_04 = XGBClassifier(
    **fixed_params_xgb,
    **study_xgb_no_calc_te_loaded.best_params
)

start = time.time()
L_xgb_opt_04.fit(
    x_train_no_calc_te, y_train, 
    eval_set=[(x_val_no_calc_te, y_val)], 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_xgb_opt_04, x_train_no_calc_te, y_train, x_val_no_calc_te, y_val, train_times[name], best_iter=L_xgb_opt_04.best_iteration)
)

[0]	validation_0-auc:0.58953
[250]	validation_0-auc:0.63782
[500]	validation_0-auc:0.64021
[665]	validation_0-auc:0.64006


In [38]:
pd.DataFrame(results)

,model_idx,auc_train,auc_val,gini,delta_auc,best_iter,trainingszeit
0,L_xgb_opt_03,0.712847,0.637973,0.275947,0.074874,782,62.368462
1,L_xgb_opt_04,0.685711,0.640470,0.280939,0.045242,565,25.698286


## Save Models

In [39]:
Modelle = [
    (L_xgb_opt_03, "L_xgb_opt_03"),
    (L_xgb_opt_04, "L_xgb_opt_04")
]

for modell, name in Modelle:
    modell.save_model(f"{name}.json")

## Notizen


- 105 aus max one hot kann raus weil target encoding 